# Ensemble Affinity Adjudication Analysis

This notebook compares benchmark model scores with final adjudication outputs in two ordered pipelines:

1. **RA analysis** runs first from the RA merged and adjudicated files.
2. **PRP analysis** runs second only when both PRP input files are present.

Figures are shown inline and no summary CSVs are created.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px

latex_textwidth_pt = 516.0 # double column # use this command in latex: \the\textwidth
# latex_textwidth_pt = 452.0 # single column
scale = 1
FIGURE_DIM = (latex_textwidth_pt * scale,latex_textwidth_pt * scale*0.6)

In [2]:
# Shared configuration and schema alignment
ignore_models = [
    'smollm2-1.7b',
    'gemma3-4b',
    'phi3-3.8b',
    'deepseek-v3.2-cloud',
    'qwen3.5-397b-cloud',
]
run_id = 'tpwrs_2026_tpwrs_2026'  # Set to None to use the latest run folder.

repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
bench_root = repo_root / 'data' / 'results' / 'affinity_benchmark'

if run_id is None:
    run_dirs = sorted(path for path in bench_root.iterdir() if path.is_dir())
    if not run_dirs:
        raise FileNotFoundError(f'No benchmark runs found under {bench_root}')
    run_dir = run_dirs[-1]
else:
    run_dir = bench_root / run_id

def append_final_rows(base_df, adjudication_df, cosine_column=None):
    column_renames = {
        'Final_Affinity': 'LLM_Affinity',
        'Final_Reason': 'LLM_Affinity_Reason',
    }
    if cosine_column is not None:
        column_renames['Cosine_x100_Mean'] = cosine_column

    final_rows = adjudication_df.copy().rename(columns=column_renames)
    provided_slug = final_rows['Model_Slug'].astype('string')
    final_method = final_rows['Final_Method'].fillna('adjudicator').astype(str)
    final_rows['Model_Slug'] = np.where(
        provided_slug.notna(),
        provided_slug.astype(str) + '_adjudicator',
        final_method,
    )

    for column in base_df.columns:
        if column not in final_rows.columns:
            final_rows[column] = pd.NA

    return pd.concat([base_df, final_rows[base_df.columns]], ignore_index=True)

print('Run directory:', run_dir)

Run directory: /mnt/c/Users/pgallego/OneDrive - Imperial College London/PhD/Repositories/article-classifier/data/results/affinity_benchmark/tpwrs_2026_tpwrs_2026


# 1. RA Analysis

Load and prepare RA benchmark and final rows, then analyze score distributions and the highest- and lowest-agreement abstract-question pairs.

In [3]:
# RA input and processing
ra_path = run_dir / 'merged_ra_affinities.csv'
ra_adjudication_path = run_dir / 'adjudicated_ra_affinities.csv'

missing_ra_inputs = [
    path for path in (ra_path, ra_adjudication_path) if not path.exists()
]
if missing_ra_inputs:
    raise FileNotFoundError(
        'Missing required RA input(s): ' + ', '.join(str(path) for path in missing_ra_inputs)
)

ra_df = pd.read_csv(ra_path)
ra_df = ra_df[~ra_df['Model_Slug'].isin(ignore_models)].copy()
ra_benchmark_models = set(ra_df['Model_Slug'].dropna().astype(str))
ra_adjudication_df = pd.read_csv(ra_adjudication_path)
ra_df = append_final_rows(
    ra_df,
    ra_adjudication_df,
    cosine_column='Cosine_x100',
)

mapping_path = repo_root / 'data' / 'processed' / 'ra_grouping_ra2025.csv'
ra_grouping_df = pd.read_csv(mapping_path)
required_mapping_cols = {'RA2025', 'Grouping'}
missing_mapping_cols = required_mapping_cols - set(ra_grouping_df.columns)
if missing_mapping_cols:
    raise ValueError(f'Missing required mapping columns: {missing_mapping_cols}')

ra_to_prp_map = (
    ra_grouping_df[['RA2025', 'Grouping']]
    .dropna(subset=['RA2025', 'Grouping'])
    .drop_duplicates()
    .rename(columns={'RA2025': 'RA2025_ID', 'Grouping': 'PRP_Name'})
)
ra_to_prp_map['RA2025_ID'] = pd.to_numeric(
    ra_to_prp_map['RA2025_ID'],
    errors='coerce',
).astype('Int64')
ra_df['RA2025_ID'] = pd.to_numeric(ra_df['RA2025_ID'], errors='coerce').astype('Int64')
ra_df = ra_df.merge(ra_to_prp_map, on='RA2025_ID', how='left')

print('RA rows:', len(ra_df), '| Comparison series:', ra_df['Model_Slug'].nunique())
display(ra_df.head(3))

RA rows: 103104 | Comparison series: 6


/tmp/ipykernel_92422/294185345.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([base_df, final_rows[base_df.columns]], ignore_index=True)


,Abstract_Index,Document Title,RA2025_ID,RA_Question,Cosine_x100,LLM_Affinity,LLM_Affinity_Reason,Was_Repaired,Repair_Method,Repair_Attempt,Repaired_At_UTC,Model_Slug,Run_ID,PRP_Name
0,10.1109/TPWRS.2025.3560178,Resilience-Constrained Economic Dispatch With ...,46,"what additional methods, tools, and metrics ar...",63.3282,75.0,Directly addresses resilience incorporation in...,False,NaN,NaN,NaN,claude-haiku-4.5,tpwrs_2026_tpwrs_2026,Planning
1,10.1109/TPWRS.2025.3560178,Resilience-Constrained Economic Dispatch With ...,47,how do system operators adequately account for...,50.1143,65.0,Addresses system impact of extreme events in o...,False,NaN,NaN,NaN,claude-haiku-4.5,tpwrs_2026_tpwrs_2026,Planning
2,10.1109/TPWRS.2025.3560178,Resilience-Constrained Economic Dispatch With ...,20,how can grid topology be flexibly adapted to b...,48.5546,25.0,No overlap with grid topology adaptation or ou...,False,NaN,NaN,NaN,claude-haiku-4.5,tpwrs_2026_tpwrs_2026,CROF


## RA Distributions and Agreement Cases

In [4]:
fig = px.box(
    ra_df,
    x='RA2025_ID',
    y='LLM_Affinity',
    color='Model_Slug',
    title='RA Affinity Distribution by Question and Comparison Series',
)
fig.update_layout(xaxis_title='RA Question', yaxis_title='LLM Affinity')
fig.show()

top_ra_disagreement_ids = (
    ra_adjudication_df.loc[ra_adjudication_df['agreement_mean'] < 1]
    .nlargest(20, 'Affinity_Range')['Abstract_Index']
    .drop_duplicates()
    .head(5)
)
high_ra_agreement_ids = (
    ra_adjudication_df.loc[ra_adjudication_df['agreement_mean'] == 1]
    .nsmallest(20, 'Affinity_Range')['Abstract_Index']
    .drop_duplicates()
    .head(5)
)

for abstract_ids, title in [
    (top_ra_disagreement_ids, 'Highest-Disagreement RA Cases'),
    (high_ra_agreement_ids, 'Highest-Agreement RA Cases'),
]:
    fig = px.scatter(
        ra_df[ra_df['Abstract_Index'].isin(abstract_ids)],
        x='RA2025_ID',
        y='LLM_Affinity',
        color='Model_Slug',
        facet_col='Abstract_Index',
        hover_data=['Model_Slug', 'PRP_Name'],
        title=title,
    )
    fig.update_layout(xaxis_title='RA Question', yaxis_title='LLM Affinity')
    fig.show()

# 2. PRP Analysis

Load, prepare, and analyze PRP benchmark and final rows only when both required PRP files are available.

In [5]:
# PRP input and processing
prp_path = run_dir / 'merged_prp_affinities.csv'
prp_adjudication_path = run_dir / 'adjudicated_prp_affinities.csv'
prp_available = prp_path.exists() and prp_adjudication_path.exists()

if prp_available:
    prp_df = pd.read_csv(prp_path)
    prp_df = prp_df[~prp_df['Model_Slug'].isin(ignore_models)].copy()
    prp_benchmark_models = set(prp_df['Model_Slug'].dropna().astype(str))
    prp_adjudication_df = pd.read_csv(prp_adjudication_path)
    prp_df = append_final_rows(prp_df, prp_adjudication_df)

    print('PRP rows:', len(prp_df), '| Comparison series:', prp_df['Model_Slug'].nunique())
    display(prp_df.head(3))
else:
    missing_prp_inputs = [
        str(path) for path in (prp_path, prp_adjudication_path) if not path.exists()
    ]
    print('PRP analysis skipped. Missing input(s):')
    for missing_path in missing_prp_inputs:
        print('-', missing_path)

PRP analysis skipped. Missing input(s):
- /mnt/c/Users/pgallego/OneDrive - Imperial College London/PhD/Repositories/article-classifier/data/results/affinity_benchmark/tpwrs_2026_tpwrs_2026/merged_prp_affinities.csv
- /mnt/c/Users/pgallego/OneDrive - Imperial College London/PhD/Repositories/article-classifier/data/results/affinity_benchmark/tpwrs_2026_tpwrs_2026/adjudicated_prp_affinities.csv


In [26]:
import sys

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.analysis.affinity_levels import affinity_level_fixed

selected_ra_abstract = top_ra_disagreement_ids.iloc[-1]
base_ra_df = ra_df[
    (ra_df['Abstract_Index'] == selected_ra_abstract)
    & (ra_df['Model_Slug'].isin(ra_benchmark_models))
]
final_ra_df = ra_df[
    (ra_df['Abstract_Index'] == selected_ra_abstract)
    & (~ra_df['Model_Slug'].isin(ra_benchmark_models))
]

programme_colours = {
    'IBR': '#4f8b75',
    'Planning': '#9b6fb6',
    'Stability': '#d96c7c',
    'CROF': '#3c9fca',
    'System Services': '#78ae68',
    'DER': '#d68a4a',
}

fig = px.box(
    base_ra_df,
    x='RA2025_ID',
    y='LLM_Affinity',
    points='all',
    color='PRP_Name',
    color_discrete_map=programme_colours,
    category_orders={
        'PRP_Name': ra_to_prp_map['PRP_Name'].drop_duplicates().tolist()
    },
    title=f'RA Affinity Distribution for {selected_ra_abstract}',
)

for band_start in range(0, 120, 20):
    band_name = affinity_level_fixed(band_start, six_bands=True)
    fig.add_hline(
        y=band_start,
        line_color='gray',
        line_width=1,
        line_dash='dash',
        # layer='below',
        annotation_text=band_name,
        annotation_position='top right',
    )

fig.add_scatter(
    x=final_ra_df['RA2025_ID'],
    y=final_ra_df['LLM_Affinity'],
    mode='markers',
    name='Mean',
    marker=dict(size=6, symbol='x', color='black'),
    hovertext=final_ra_df['PRP_Name'],
    hovertemplate='RA2025_ID=%{x}<br>LLM_Affinity=%{y}<br>PRP=%{hovertext}<extra></extra>',
)
fig.update_layout(
    xaxis_title='RA Question',
    yaxis_title='LLM Affinity',
    width=FIGURE_DIM[0],
    height=FIGURE_DIM[1],
    xaxis=dict(range=[-1, 53]),
    yaxis=dict(range=[-4, 120], dtick=20),
    title=dict(y=0.99, yanchor='top'),
    legend=dict(title=None, orientation='h', yanchor='bottom', y=1, xanchor='center', x=0.5),
)

n = 4
fig.update_layout(margin=dict(l=n, r=n+30, t=n+65, b=n+50))
fig.show()

## PRP Distributions and Agreement Cases

In [7]:
if prp_available:
    fig = px.box(
        prp_df,
        x='PRP_Name',
        y='LLM_Affinity',
        color='Model_Slug',
        title='PRP Affinity Distribution by Programme and Comparison Series',
    )
    fig.update_layout(xaxis_title='PRP Name', yaxis_title='LLM Affinity')
    fig.show()

    top_prp_disagreement_ids = (
        prp_adjudication_df.loc[prp_adjudication_df['agreement_mean'] < 1]
        .nlargest(20, 'Affinity_Range')['Abstract_Index']
        .drop_duplicates()
        .head(5)
    )
    high_prp_agreement_ids = (
        prp_adjudication_df.loc[prp_adjudication_df['agreement_mean'] == 1]
        .nsmallest(20, 'Affinity_Range')['Abstract_Index']
        .drop_duplicates()
        .head(5)
    )

    for abstract_ids, title in [
        (top_prp_disagreement_ids, 'Highest-Disagreement PRP Cases'),
        (high_prp_agreement_ids, 'Highest-Agreement PRP Cases'),
    ]:
        fig = px.scatter(
            prp_df[prp_df['Abstract_Index'].isin(abstract_ids)],
            x='PRP_Name',
            y='LLM_Affinity',
            color='Model_Slug',
            facet_col='Abstract_Index',
            title=title,
        )
        fig.update_layout(yaxis_title='LLM Affinity')
        fig.show()
else:
    print('PRP distribution and agreement plots skipped.')

PRP distribution and agreement plots skipped.
